In [6]:
!pip install datasets

In [6]:
!pip install evaluate

In [3]:
from datasets import load_dataset
import evaluate
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import numpy as np

W0611 21:42:50.063000 37400 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [7]:
import huggingface_hub
print(huggingface_hub.__version__)

0.36.0


In [8]:
import datasets
print(datasets.__version__)

2.16.1


In [9]:
import transformers
print(transformers.__version__)



4.37.2


In [10]:
!pip uninstall -y huggingface_hub
!pip install huggingface_hub==0.36.0

Found existing installation: huggingface-hub 0.36.0
Uninstalling huggingface-hub-0.36.0:
  Successfully uninstalled huggingface-hub-0.36.0
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
Using cached huggingface_hub-0.36.0-py3-none-any.whl (566 kB)


In [11]:
import transformers
import huggingface_hub

print(transformers.__version__)
print(huggingface_hub.__version__)

4.37.2
0.36.0


In [12]:
import datasets
print(datasets.__version__)

2.16.1


In [13]:
from datasets import load_dataset

dataset = load_dataset("cnn_dailymail", "3.0.0")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [14]:
dataset['train'].features

{'article': Value(dtype='string', id=None),
 'highlights': Value(dtype='string', id=None),
 'id': Value(dtype='string', id=None)}

In [15]:
# Taking the subset of the dataset for the finetuning purpose
train_subset = dataset["train"].select(range(10000))
validation_subset = dataset["validation"].select(range(1000))
test_subset = dataset["test"].select(range(1000))

In [16]:
checkpoint = 't5-small'
tokenizer = T5Tokenizer.from_pretrained(checkpoint)

C:\Users\Radhika\.conda\envs\newenv\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [17]:
def preprocess_function(examples):
    inputs = ["summarize: " + article for article in examples["article"]]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["highlights"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [18]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

In [19]:
# Tokenize datasets
tokenized_train = train_subset.map(preprocess_function, batched=True)
tokenized_validation = validation_subset.map(preprocess_function, batched=True)
tokenized_test = test_subset.map(preprocess_function, batched=True)

In [20]:
# Load Model
model = T5ForConditionalGeneration.from_pretrained(checkpoint)

In [21]:
# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [22]:
import numpy as np
rouge = evaluate.load("rouge")

def compute_metrics(pred):

    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    pred_ids = np.where(
        pred_ids != -100,
        pred_ids,
        tokenizer.pad_token_id
    )

    labels_ids = np.where(
        labels_ids != -100,
        labels_ids,
        tokenizer.pad_token_id
    )

    pred_str = tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = tokenizer.batch_decode(
        labels_ids,
        skip_special_tokens=True
    )

    rouge_output = rouge.compute(
        predictions=pred_str,
        references=label_str,
        use_stemmer=True
    )

    result = {
        key: value * 100
        for key, value in rouge_output.items()
    }

    prediction_lens = [
        np.count_nonzero(pred != tokenizer.pad_token_id)
        for pred in pred_ids
    ]

    result["gen_len"] = np.mean(prediction_lens)

    return result

In [23]:
# Seq2Seq training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=150,
    generation_num_beams=4,
    fp16=True
)

In [24]:

## Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,                       # The model to be trained
    args=training_args,                # Training arguments defined with Seq2SeqTrainingArguments
    train_dataset=tokenized_train,     # The training dataset
    eval_dataset=tokenized_validation, # The evaluation dataset
    data_collator=data_collator,       # The data collator for processing data batches
    tokenizer=tokenizer,               # The tokenizer used for preprocessing
    compute_metrics=compute_metrics,   # The function to compute evaluation metrics
)



C:\Users\Radhika\.conda\envs\newenv\lib\site-packages\accelerate\accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [25]:
import accelerate
print(accelerate.__version__)

0.27.2


In [26]:
!pip uninstall -y accelerate
!pip install accelerate==0.27.2

Found existing installation: accelerate 0.27.2
Uninstalling accelerate-0.27.2:
  Successfully uninstalled accelerate-0.27.2
  Using cached accelerate-0.27.2-py3-none-any.whl.metadata (18 kB)
Using cached accelerate-0.27.2-py3-none-any.whl (279 kB)


In [27]:
## Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,                       # The model to be trained
    args=training_args,                # Training arguments defined with Seq2SeqTrainingArguments
    train_dataset=tokenized_train,     # The training dataset
    eval_dataset=tokenized_validation, # The evaluation dataset
    data_collator=data_collator,       # The data collator for processing data batches
    tokenizer=tokenizer,               # The tokenizer used for preprocessing
    compute_metrics=compute_metrics,   # The function to compute evaluation metrics
)

In [28]:
import os

print(os.listdir("./results"))

['checkpoint-500']


In [30]:
# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,2.207000,2.157441,30.700310,11.753146,22.070360,22.092890,76.869000
2,2.135700,2.154184,30.959434,11.802871,22.133963,22.142554,75.490000
3,2.120000,2.154569,31.059558,11.875777,22.216478,22.210545,74.655000


Checkpoint destination directory ./results\checkpoint-500 already exists and is non-empty.Saving will proceed but saved results may be invalid.


TrainOutput(global_step=1875, training_loss=2.1457508138020835, metrics={'train_runtime': 1535.0253, 'train_samples_per_second': 19.544, 'train_steps_per_second': 1.221, 'total_flos': 4060254044160000.0, 'train_loss': 2.1457508138020835, 'epoch': 3.0})

In [29]:
# Evaluate the model on validation set
trainer.evaluate()

# Evaluate the model on test set
test_results = trainer.evaluate(eval_dataset=tokenized_test)

print(test_results)

{'eval_loss': 2.5619618892669678, 'eval_rouge1': 30.200884483113622, 'eval_rouge2': 11.30267796935787, 'eval_rougeL': 21.883569686637745, 'eval_rougeLsum': 21.882491205164428, 'eval_gen_len': 70.24, 'eval_runtime': 232.2903, 'eval_samples_per_second': 4.305, 'eval_steps_per_second': 0.271}


In [33]:
import torch
# Select a specific data point from the test dataset
test_index = 78  # Change this index to the specific data point you want to summarize
example_text = dataset["test"][test_index]["article"]

# Preprocess the input text
input_text = "summarize: " + example_text
inputs = tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = inputs.to(device)
# Generate the summary
summary_ids = model.generate(inputs, max_length=150, min_length=40, length_penalty=2.0, num_beams=4, early_stopping=True)

# Decode the generated summary
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Original Text:\n", example_text)
print("\nGenerated Summary:\n", summary)

Original Text:
 (CNN)The nation's top stories will be unfolding Tuesday in courthouses and political arenas across the country. Massachusetts is hosting two of the highest-profile court trials in recent memory -- those of former New England Patriot Aaron Hernandez and Boston bombing suspect Dzhokhar Tsarnaev. Both lengthy trials are coming to a close. In Louisville, Kentucky, Sen. Rand Paul made the not-so-surprising announcement that he will run for president, while in Chicago, voters will head to the polls in a very surprising runoff between Mayor Rahm Emanuel and challenger Jesus "Chuy" Garcia. And in Ferguson, Missouri, the shadow of Michael Brown and the protests over his shooting by Officer Darren Wilson will loom large over the city's elections. Here's a breakdown of what to expect today and how we got here: . Tsarnaev, who's accused of detonating a bomb at the 2013 Boston Marathon along with his now-deceased brother, faces the stiffest of penalties -- life in prison or the deat

In [34]:
model.save_pretrained("./my_t5_summarizer")
tokenizer.save_pretrained("./my_t5_summarizer")

('./my_t5_summarizer\\tokenizer_config.json',
 './my_t5_summarizer\\special_tokens_map.json',
 './my_t5_summarizer\\spiece.model',
 './my_t5_summarizer\\added_tokens.json')

In [35]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

model = T5ForConditionalGeneration.from_pretrained(
    "./my_t5_summarizer"
)

tokenizer = T5Tokenizer.from_pretrained(
    "./my_t5_summarizer"
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [36]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print(next(model.parameters()).device)

cuda:0


In [37]:
import torch

example_text = """India secured a memorable victory over Australia in the final of a major international cricket tournament. Chasing a target of 289 runs, India lost an early wicket but quickly recovered through a strong partnership between Virat Kohli and Shubman Gill.

Kohli played a patient innings, rotating the strike effectively and punishing loose deliveries. Gill complemented him with aggressive stroke play, keeping the required run rate under control. The pair added more than 140 runs for the second wicket before Gill was dismissed.

Australia attempted a comeback through their fast bowlers, who picked up wickets in the middle overs. However, KL Rahul and Hardik Pandya stabilized the innings and ensured that India remained ahead of the required rate.

The victory sparked celebrations across the country, with fans praising the team's consistency throughout the tournament. Captain Rohit Sharma credited the bowling unit for restricting Australia to a manageable total and highlighted the team's ability to perform under pressure.

Cricket analysts described the match as one of India's most complete performances in recent years. The win strengthened India's position as one of the leading teams in world cricket and boosted confidence ahead of upcoming international competitions.
"""

input_text = "summarize: " + example_text

inputs = tokenizer.encode(
    input_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = inputs.to(device)

summary_ids = model.generate(
    inputs,
    max_length=150,
    min_length=40,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Original Text:\n", example_text)
print("\nGenerated Summary:\n", summary)

Original Text:
 India secured a memorable victory over Australia in the final of a major international cricket tournament. Chasing a target of 289 runs, India lost an early wicket but quickly recovered through a strong partnership between Virat Kohli and Shubman Gill.

Kohli played a patient innings, rotating the strike effectively and punishing loose deliveries. Gill complemented him with aggressive stroke play, keeping the required run rate under control. The pair added more than 140 runs for the second wicket before Gill was dismissed.

Australia attempted a comeback through their fast bowlers, who picked up wickets in the middle overs. However, KL Rahul and Hardik Pandya stabilized the innings and ensured that India remained ahead of the required rate.

The victory sparked celebrations across the country, with fans praising the team's consistency throughout the tournament. Captain Rohit Sharma credited the bowling unit for restricting Australia to a manageable total and highlighted

In [38]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print(next(model.parameters()).device)

cuda:0


In [1]:
import os
print(os.getcwd())

C:\Users\Radhika
